# Data Profiling & Quality Risk Analysis
## Workshop-2: Building a Reliable Batch Data Pipeline
**Course:** ETL (G01) — Faculty of Engineering and Basic Sciences (UAO)  
**Objective:** Systematic evidence gathering on raw input sources (Spotify and Grammy Awards) to justify Great Expectations quality rules, validation gates, and engineering transformation logic.

---
### Minimum Analysis Categories (Section 6.3):
1. **Structure:** Schemas, columns, row counts, data types.
2. **Completeness:** Missing-value counts and percentages interpreted in context.
3. **Uniqueness and Duplication:** Candidate keys, duplicate patterns, and grain analysis.
4. **Categorical Content:** Cardinality, unexpected labels, frequency patterns.
5. **Numerical Content:** Ranges, distributions, impossible values, extreme outliers.
6. **Temporal Content:** Date coverage, parseability, temporal gaps.
7. **Cross-source Consistency:** Integration contract matching feasibility.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Profiling dependencies loaded successfully.")


## 1. Raw Data Ingestion

In [ ]:
# Ingest datasets
df_spotify = pd.read_csv("../spotify_dataset.csv")
df_grammy = pd.read_csv("../the_grammy_awards.csv")

print(f"Spotify Dimensions: {df_spotify.shape[0]:,} rows, {df_spotify.shape[1]} columns")
print(f"Grammys Dimensions: {df_grammy.shape[0]:,} rows, {df_grammy.shape[1]} columns")


## 2. Structure & Data Types Analysis

In [ ]:
print("=== SPOTIFY STRUCTURE ===")
display(df_spotify.dtypes.to_frame("Data Type"))

print("\n=== GRAMMYS STRUCTURE ===")
display(df_grammy.dtypes.to_frame("Data Type"))


## 3. Completeness Analysis (Missing Values)

In [ ]:
# Spotify missing values
null_spotify = pd.DataFrame({
    "Null Count": df_spotify.isnull().sum(),
    "Null %": (df_spotify.isnull().sum() / len(df_spotify)) * 100
})
print("Spotify Missing Values:")
display(null_spotify[null_spotify["Null Count"] > 0])

# Grammys missing values
null_grammy = pd.DataFrame({
    "Null Count": df_grammy.isnull().sum(),
    "Null %": (df_grammy.isnull().sum() / len(df_grammy)) * 100
})
print("Grammys Missing Values:")
display(null_grammy[null_grammy["Null Count"] > 0])


## 4. Uniqueness, Candidate Keys & Grain Analysis

In [ ]:
print("=== SPOTIFY UNIQUENESS ===")
total_rows = len(df_spotify)
unique_tracks = df_spotify["track_id"].nunique()
dup_track_ids = df_spotify["track_id"].duplicated().sum()
dup_track_genre = df_spotify.duplicated(subset=["track_id", "track_genre"]).sum()

print(f"Total Rows: {total_rows:,}")
print(f"Unique track_ids: {unique_tracks:,}")
print(f"Rows with repeated track_id: {dup_track_ids:,} (tracks appearing in multiple genres)")
print(f"Exact Duplicate Tuples (track_id, track_genre): {dup_track_genre}")

print("\n=== GRAMMYS UNIQUENESS ===")
dup_grammy_natural = df_grammy.duplicated(subset=["year", "category", "nominee"]).sum()
print(f"Natural Key Duplicates (year, category, nominee): {dup_grammy_natural}")


## 5. Categorical Content Analysis

In [ ]:
print(f"Total distinct Spotify genres: {df_spotify['track_genre'].nunique()}")
print(f"Top 10 Genres by Track Count:\n{df_spotify['track_genre'].value_counts().head(10)}")

print(f"\nTotal Grammy Award Categories: {df_grammy['category'].nunique()}")
print(f"Winner Indicator Distribution:\n{df_grammy['winner'].value_counts()}")


## 6. Numerical Content & Boundary Analysis

In [ ]:
num_cols = ["popularity", "duration_ms", "danceability", "energy", "loudness", "valence", "tempo"]
display(df_spotify[num_cols].describe().round(3))

# Detection of physically impossible values
zero_duration = (df_spotify["duration_ms"] <= 1000).sum()
zero_tempo = (df_spotify["tempo"] == 0).sum()
print(f"Tracks with duration <= 1,000 ms: {zero_duration}")
print(f"Tracks with tempo == 0 BPM: {zero_tempo}")


## 7. Temporal Content Analysis

In [ ]:
print(f"Grammy Year Range: {df_grammy['year'].min()} to {df_grammy['year'].max()}")
print(f"Total award years covered: {df_grammy['year'].nunique()}")

# Parsing ISO timestamps with mixed offsets
df_grammy["pub_parsed"] = pd.to_datetime(df_grammy["published_at"], utc=True, errors="coerce")
invalid_dates = df_grammy["pub_parsed"].isnull().sum()
print(f"Unparseable timestamps in published_at: {invalid_dates}")


## 8. Cross-Source Consistency & Integration Matching

In [ ]:
# Exact title match baseline test
spotify_titles = df_spotify["track_name"].dropna().str.lower().str.strip()
grammy_nominees = df_grammy["nominee"].dropna().str.lower().str.strip()

exact_title_matches = len(set(spotify_titles).intersection(set(grammy_nominees)))
print(f"Direct song title matches: {exact_title_matches:,}")

# Artist formatting difference demonstration
print("\nArtist format difference:")
print("Spotify:", df_spotify["artists"].iloc[2])
print("Grammys:", df_grammy[df_grammy["artist"].notnull()]["artist"].iloc[0])


## 9. Profiling Synthesis & Quality Risk Matrix

| Dataset / Attribute | Observed Profiling Evidence | Potential Quality Risk | Severity Level | Related Requirement |
| :--- | :--- | :--- | :--- | :--- |
| `spotify.track_id` | 450 exact duplicate rows on `(track_id, track_genre)`. | Silent double-counting and aggregation distortion in DW. | **Critical** (Raw Gate) | RQ-01, RQ-02 |
| `spotify.duration_ms` | Corrupted row detected with duration = 0 ms and NaN metadata. | Distorted average duration and audio metrics. | **Critical** (Raw Gate) | RQ-03 |
| `spotify.audio_features` | Audio characteristics must lie within $[0.0, 1.0]$. | Correlation distortion in perceptual radar analysis. | **Critical** (Raw Gate) | RQ-01 |
| `spotify.popularity` | Popularity ranges $[0, 100]$. Metadata nulls break FK relations. | Broken foreign keys and dimension integrity. | **Critical** (Raw Gate) | RQ-02 |
| `grammy.year` | Range $[1958, 2019]$. Out-of-bounds years break time dimension. | Corrupted temporal hierarchies and time series trends. | **Critical** (Raw Gate) | RQ-01, RQ-03 |
| `grammy.nominee` | 6 rows with null `nominee` in historical craft categories. | Inability to establish catalog join. | **Warning** (Raw Gate) | RQ-01, RQ-02 |
| `integrated.unmatched` | Most Spotify catalog tracks do not have Grammy awards. | False positive if 100% match expected; modeled with `is_grammy_winner`. | **Informational** (Prepared Gate) | RQ-01, RQ-02 |
| `prepared.audio_bounds` | Post-transformation audio features must preserve $[0.0, 1.0]$. | Unloadable dimensional rows. | **Critical** (Prepared Gate) | RQ-01 |
